# 07 — Model Testing & Visualization (CNN + YOLO11)

**Model Lead follow-up.** Blind-test visualizations for both published models, pulled fresh from Hugging Face — no retraining. Predictions are shown with clear, non-overlapping labels (numbered markers + a text legend, not overlapping arrows) and, for YOLO11, IoU against ground truth so localization quality is visible, not just the class label.

| Section | What happens |
|---|---|
| 0 | Setup |
| 1 | Load both trained models from Hugging Face |
| 2 | CNN — 5 random test images, predicted vs. true label, confidence shown |
| 3 | YOLO11 — 5 random test images, numbered boxes + legend, confidence + IoU |
| 4 | Push both result images to Hugging Face |

## 0. Setup

In [ ]:
!git clone https://github.com/neuroarcane/dental-cavity-detector.git
%cd dental-cavity-detector
!pip install -r requirements.txt
!nvidia-smi

In [ ]:
from pathlib import Path

data_root = Path('/content/dental_data')
data_root.mkdir(parents=True, exist_ok=True)
!cp -r "data/raw/Dental X-ray.v1i.yolov11" {data_root}/
!cp -r "data/raw/Dental X-Ray Panoramic Dataset" {data_root}/
(data_root / "Dental X-ray.v1i.yolov11.zip.extracted").touch()
(data_root / "Dental X-Ray Panoramic Dataset.zip.extracted").touch()

from src.data.prepare_dataset import merge_datasets, print_merge_summary
from src.data.split import remove_exact_duplicates, stratified_resplit
from src.data.balance import oversample_minority_classes

merge_datasets()
remove_exact_duplicates()
stratified_resplit(train_frac=0.7, valid_frac=0.15, test_frac=0.15)
oversample_minority_classes(split='train', target_ratio=0.5, max_duplicates_per_image=5)

!cat data/processed/data.yaml

## 1. Load both trained models from Hugging Face

In [ ]:
import yaml, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from ultralytics import YOLO


In [ ]:
from huggingface_hub import hf_hub_download, HfApi, login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
HF_REPO_ID = 'aparnamohankumar/dental-cavity-detector'
api = HfApi()

In [ ]:
CLASS_NAMES = ['Cavity', 'Filling', 'Crown', 'Impacted Tooth']
IMG_SIZE = 128
DATA_DIR = Path('data/processed')

cnn_path = hf_hub_download(repo_id=HF_REPO_ID, filename='cnn_baseline.keras')
cnn_model = tf.keras.models.load_model(cnn_path)

yolo_path = hf_hub_download(repo_id=HF_REPO_ID, filename='yolo11_baseline_best.pt')
yolo_model = YOLO(yolo_path)

print('Both models loaded.')

In [ ]:
with open(DATA_DIR / 'data.yaml') as f:
    data_yaml = yaml.safe_load(f)
yaml_class_names = data_yaml['names']
PRIORITY = ['Cavity', 'Crown', 'Impacted Tooth', 'Filling']

def build_singlelabel_index(split_dir):
    images_dir, labels_dir = split_dir / 'images', split_dir / 'labels'
    rows = []
    for img_path in sorted(images_dir.glob('*.*')):
        label_path = labels_dir / f'{img_path.stem}.txt'
        present = set()
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    present.add(yaml_class_names[int(line.split()[0])])
        chosen = PRIORITY[-1]
        for cls in PRIORITY:
            if cls in present:
                chosen = cls
                break
        rows.append({'path': str(img_path), 'class_id': CLASS_NAMES.index(chosen)})
    return pd.DataFrame(rows)

test_df = build_singlelabel_index(DATA_DIR / 'test')
print('Test set:', len(test_df), 'images')

## 2. CNN — 5 random blind test images

Predicted label, true label, and confidence — titles sit in dedicated space above each image, so nothing overlaps regardless of label length.

In [ ]:
sample_df = test_df.sample(5, random_state=1).reset_index(drop=True)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, ax in enumerate(axes):
    row = sample_df.iloc[i]
    img_raw = tf.io.read_file(row['path'])
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img_resized = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0

    pred_probs = cnn_model.predict(tf.expand_dims(img_resized, 0), verbose=False)
    pred_class = CLASS_NAMES[np.argmax(pred_probs)]
    confidence = np.max(pred_probs)
    true_class = CLASS_NAMES[row['class_id']]

    ax.imshow(img_resized)
    ax.axis('off')
    ax.set_title(f'Pred: {pred_class} ({confidence:.0%})\nTrue: {true_class}', fontsize=11, pad=8)

plt.tight_layout()
plt.savefig('cnn_blind_test_5.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. YOLO11 — 5 random blind test images, with IoU

Numbered markers on the image, full legend (class, confidence, IoU vs. ground truth) as plain text underneath — no arrows, nothing that can visually overlap. Orange dashed boxes are ground truth; cyan solid boxes are the model's predictions.

In [ ]:
def load_gt_boxes(img_path, img_w, img_h):
    label_path = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if line.strip():
                cls_id, xc, yc, bw, bh = map(float, line.split())
                x1 = (xc - bw / 2) * img_w
                y1 = (yc - bh / 2) * img_h
                x2 = (xc + bw / 2) * img_w
                y2 = (yc + bh / 2) * img_h
                boxes.append({'cls': int(cls_id), 'box': [x1, y1, x2, y2]})
    return boxes

def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

In [ ]:
sample_paths = random.sample(list((DATA_DIR / 'test' / 'images').glob('*.*')), 5)

fig, axes = plt.subplots(2, 5, figsize=(24, 9), gridspec_kw={'height_ratios': [4, 1.2]})
all_ious = []

for col, img_path in enumerate(sample_paths):
    result = yolo_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    gt_boxes = load_gt_boxes(img_path, w, h)

    ax_img = axes[0, col]
    ax_img.imshow(img)
    ax_img.axis('off')

    for gt in gt_boxes:
        gx1, gy1, gx2, gy2 = gt['box']
        ax_img.add_patch(plt.Rectangle((gx1, gy1), gx2-gx1, gy2-gy1, fill=False,
                                        edgecolor='orange', linewidth=1.3, linestyle='--'))

    boxes = result.boxes
    legend_lines = []
    image_ious = []

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_name = CLASS_NAMES[int(box.cls[0])]
        conf = float(box.conf[0])

        best_iou = max((compute_iou([x1, y1, x2, y2], gt['box']) for gt in gt_boxes), default=0.0)
        image_ious.append(best_iou)
        all_ious.append(best_iou)

        ax_img.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='cyan', linewidth=1.5))
        ax_img.text(x1, y1 - 4, str(i+1), fontsize=11, fontweight='bold', color='black',
                    bbox=dict(boxstyle='circle,pad=0.15', fc='cyan', ec='none'))

        legend_lines.append(f'{i+1}. {cls_name} \u2014 conf {conf:.0%} \u2014 IoU {best_iou:.2f}')

    mean_iou = np.mean(image_ious) if image_ious else 0.0
    ax_img.set_title(f'mean IoU: {mean_iou:.2f}', fontsize=11, pad=6)

    ax_legend = axes[1, col]
    ax_legend.axis('off')
    legend_text = '\n'.join(legend_lines) if legend_lines else 'No detections'
    ax_legend.text(0, 1, legend_text, fontsize=9, va='top', ha='left', family='monospace')

overall_mean_iou = np.mean(all_ious) if all_ious else 0.0
print(f'Overall mean IoU across all {len(all_ious)} detections in this sample: {overall_mean_iou:.3f}')

fig.suptitle(
    'Cyan solid box = prediction | Orange dashed box = ground truth | conf = model confidence | IoU = box overlap with ground truth (1.0 = perfect)',
    fontsize=10, y=1.02
)

plt.tight_layout()
plt.savefig('yolo_blind_test_5_iou.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Push both to Hugging Face

In [ ]:
api.upload_file(path_or_fileobj='cnn_blind_test_5.png', path_in_repo='cnn_blind_test_5.png', repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj='yolo_blind_test_5_iou.png', path_in_repo='yolo_blind_test_5_iou.png', repo_id=HF_REPO_ID)
print('Pushed both.')